# Generate Dataset Person Name for Train

In [1]:
import csv
import json
import os
import random
import re

# Label harus "PER" (bukan "PERSON") supaya cocok dengan skema label
# pipeline pretrained xx_ent_wiki_sm yang kita fine-tune (PER/ORG/LOC/MISC).
LABEL_PERSON = "PER"

# Label baru untuk alamat. TIDAK ada di skema asli xx_ent_wiki_sm, jadi ini
# akan jadi label baru (add_label) saat training - lihat penjelasan di
# person_name_ner_train.ipynb.
LABEL_ADDRESS = "ADR"

## Nama Indo

In [2]:

# 1. Kumpulan Nama Indonesia (Kaya variasi: daerah, marga, gelar, 1 kata)
nama_indonesia = [
    "Suharto",
    "Budi",
    "Joko Widodo",
    "Sri Mulyani",
    "Abdul Haris Nasution",
    "Ida Bagus Oka",
    "Siti Aminah",
    "Sukaryo",
    "Boediono",
    "Megawati Soekarnoputri",
    "Luhut Binsar Pandjaitan",
    "Raden Adjeng Kartini",
    "Pramoedya Ananta Toer",
    "Cut Nyak Dien",
    "I Gusti Ngurah Rai",
    "Abdurrahman Wahid",
    "B.J. Habibie",
    "Prabowo Subianto",
    "Anies Baswedan",
    "Ganjar Pranowo",
    "Ridwan Kamil",
    "Nadiem Makarim",
    "Susi Pudjiastuti",
    "Retno Marsudi",
    "Yasonna Laoly",
    "Agus Harimurti Yudhoyono",
    "Puan Maharani",
    "Budi Gunadi Sadikin",
    "Erick Thohir",
    "Sandiaga Uno",
    "Dian Sastrowardoyo",
    "Reza Rahadian",
    "Iko Uwais",
    "Joe Taslim",
    "Anggun C. Sasmi",
    "Agnez Mo",
    "Raisa Andriana",
    "Isyana Sarasvati",
    "Bambang Pamungkas",
    "Taufik Hidayat",
    "Susi Susanti",
    "Kevin Sanjaya Sukamuljo",
    "Greysia Polii",
    "Apriyani Rahayu",
    "Eko Yuli Irawan",
    "Tigor Silaban",
    "Andi Mallarangeng",
    "Tubagus Hasanuddin",
    "Gusti Kanjeng Ratu Hemas",
    "Rina Nose",
    "Cak Lontong",
    "Ahmad Dhani",
    "Iwan Fals",
    "Ebiet G. Ade",
    "Irfan Hakim",
    "Evry Pramudya",
    "Aji Masaid",
    "Muhammad Hasan Sadikin",
    "Muhammad Ikbal",
    "Fajar Nugraha",
    "Dini Imaniar",
    "Alif Santoso",
    "Dedi Mulyadi",
    "Karni Ilyas",
    # Nama tunggal (1 kata) sehari-hari - kategori ini sebelumnya cuma
    # diwakili "Budi", padahal nama depan tunggal sangat umum di teks bebas
    # (subjek kalimat, atribusi dialog, sapaan informal). Ditambah supaya
    # model tidak hanya hafal nama-nama tokoh publik majemuk.
    "Rina",
    "Sari",
    "Ahmad",
    "Doni",
    "Lisa",
    "Wahyu",
    "Nita",
    "Hendra",
    "Maria",
    "Tio",
    "Fitri",
    "Yoga",
    "Wati",
    "Lestari",
    "Rian",
    "Dewi",
    "Eko",
    "Agus",
    "Rudi",
    "Wawan",
    "Sinta",
    "Putri",
    "Made",
    "Komang",
    "Wayan",
    "Nyoman",
    "Yanti",
    "Dian",
    "Fajar",
    "Yudi",
]


## Nama Internasional

In [3]:

# 2. Kumpulan Nama Internasional (Berbagai benua, aksen, dan format)
nama_internasional = [
    "Elon Musk",
    "Vladimir Putin",
    "Xi Jinping",
    "Jean-Luc Picard",
    "Ruud van Nistelrooy",
    "René Descartes",
    "François Hollande",
    "John Doe",
    "Kim Jong-un",
    "Fatima binti Muhammad",
    "Leonardo DiCaprio",
    "Marie Curie",
    "Albert Einstein",
    "Guillermo del Toro",
    "Bill Gates",
    "Steve Jobs",
    "Mark Zuckerberg",
    "Jeff Bezos",
    "Warren Buffett",
    "Bernard Arnault",
    "Larry Page",
    "Sergey Brin",
    "Barack Obama",
    "Joe Biden",
    "Donald Trump",
    "Kamala Harris",
    "Emmanuel Macron",
    "Angela Merkel",
    "Boris Johnson",
    "Rishi Sunak",
    "Justin Trudeau",
    "Volodymyr Zelenskyy",
    "Shinzo Abe",
    "Narendra Modi",
    "Anthony Albanese",
    "Jacinda Ardern",
    "Nelson Mandela",
    "Mahatma Gandhi",
    "Martin Luther King Jr.",
    "Winston Churchill",
    "Franklin D. Roosevelt",
    "Keanu Reeves",
    "Scarlett Johansson",
    "Jackie Chan",
    "Bruce Lee",
    "Shah Rukh Khan",
    "Priyanka Chopra",
    "Diego Maradona",
    "Lionel Messi",
    "Cristiano Ronaldo",
    "Zinedine Zidane",
    "Serena Williams",
    "Roger Federer",
    "Usain Bolt",
    "Michael Phelps",
    "Simone Biles",
    "LeBron James",
    "Michael Jordan",
    "Kobe Bryant",
    "Taylor Swift",
    "Beyoncé",
    "Neymar Jr.",
    "Conor O'Brien",
    "Michael McDonald",
    "Anthony D'Angelo",
    "Anne-Marie Laurent",
    "Jean-Pierre Dubois",
    # Varian nama belakang saja (tanpa nama depan) - meniru gaya penulisan
    # berita yang menyebut nama lengkap sekali lalu memakai nama belakang
    # saja di kalimat berikutnya. Sengaja memakai nama belakang dari entri
    # majemuk di atas supaya model belajar mengenali keduanya sebagai PERSON.
    "O'Brien",
    "McDonald",
    "D'Angelo",
]

## Nama dengan Gelar

In [4]:

# 3. Kumpulan Nama dengan Gelar (akademis, profesi, agama, adat/kebangsawanan, militer)
nama_dengan_gelar = [
    # Gelar akademis di depan
    "Dr. Andi Wijaya",
    "Prof. Emil Salim",
    "Ir. Soekarno",
    "Drs. Muhammad Yamin",
    "Dra. Kartini Rahayu",
    "Prof. Dr. Ing. B.J. Habibie",
    "Prof. Dr. dr. Zubairi Djoerban",
    "dr. Reisa Broto Asmoro",
    "drg. Maya Kusuma",
    "drh. Bimo Prakoso",
    # Gelar akademis di belakang
    "Bambang Permadi Soemantri Brodjonegoro, M.Sc.",
    "Hartono, S.T., M.T.",
    "Yusril Ihza Mahendra, S.H., M.Sc.",
    "Kartika Sari, Sp.KG",
    "Boyke Dian Nugraha, Sp.OG",
    "Rangga Pratama, S.Kom., M.Kom.",
    "Wulan Sari, S.Pd., M.Pd.",
    "Dewi Anggraini, S.Psi., Psikolog",
    "Faisal Rahman, S.E., Ak., CA.",
    "Made Wirawan, S.H., M.H.",
    # Kombinasi gelar depan dan belakang
    "Dr. Andi Wijaya, Sp.PD",
    "Prof. Ir. Bambang Permadi Soemantri Brodjonegoro, M.Sc.",
    "Ir. Djoko Kirmanto, Dipl.HE",
    "dr. Tirta Mandira Hudhi, M.Ked.",
    "Prof. Dr. Anies Baswedan, M.P.P.",
    # Gelar keagamaan
    "K.H. Ma'ruf Amin",
    "Hj. Siti Fatimah",
    "H. Rhoma Irama",
    "Ustadz Abdul Somad",
    "Ustadzah Oki Setiana Dewi",
    "Kyai Haji Hasyim Asy'ari",
    "Buya Hamka",
    "Habib Rizieq Shihab",
    "Gus Miftah",
    "Tuan Guru Bajang",
    # Gelar adat / kebangsawanan
    "R.A. Kartini",
    "R.M. Said Prawirosoedirdjo",
    "Raden Mas Soewardi Suryaningrat",
    "Sri Sultan Hamengkubuwono X",
    "Kanjeng Gusti Pangeran Adipati Arya Mangkunegara IX",
    "Tubagus Chasan Sochib",
    "Andi Djemma",
    # Gelar militer / kepolisian
    "Jenderal Gatot Nurmantyo",
    "Letjen (Purn) Prabowo Subianto",
    "Laksamana Muda TNI Sinar Adi",
    "Brigjen Pol Listyo Sigit Prabowo",
    "AKBP Ade Safri Simanjuntak",
    "Kolonel Inf. Rizky Ramadhan",
]


## Nama yang Juga Dipakai sebagai Nama Jalan

In [5]:

# 3b. Nama tokoh yang JUGA lazim dipakai sebagai nama jalan di Indonesia.
#    Dipakai di DUA konteks: sebagai PER berdiri sendiri ("Ahmad Yani adalah
#    pahlawan revolusi") dan sebagai bagian nama jalan ("Jl. Ahmad Yani No. 5").
#    Ini sengaja dibuat tumpang tindih supaya model belajar membedakan
#    "Jl. + nama" (ALAMAT) dari nama yang sama tanpa "Jl." (PER) - lihat
#    template_kalimat (memakai nama_ganda_jalan_orang sbg PER biasa) dan
#    nama_jalan_dari_orang di bagian Alamat (memakai nama yang sama sbg jalan).
nama_ganda_jalan_orang = [
    "Ahmad Yani",
    "Gatot Subroto",
    "Basuki Rahmat",
    "Diponegoro",
    "Sudirman",
    "Ahmad Dahlan",
    "Imam Bonjol",
]


In [6]:

# 3c. Versi internasional dari nama_ganda_jalan_orang: tokoh yang namanya
#    juga lazim dipakai sebagai nama jalan di negara Barat (Washington
#    Street, Lincoln Avenue, dst.) - sumber ambiguitas yang sama, tapi
#    untuk format alamat internasional.
nama_ganda_jalan_orang_internasional = [
    "Washington",
    "Lincoln",
    "Churchill",
    "Kennedy",
    "Jefferson",
]


## Kata Bukan PERSON (Negative Examples)

In [ ]:
# 4. Kumpulan kata/frasa yang MIRIP entitas tapi BUKAN PERSON
#    (dipakai untuk negative & hard-negative example, agar model belajar
#    membedakan nama orang dari tempat, organisasi, tanggal, kontak, dsb.)
tempat = [
    "Jakarta",
    "Bandung",
    "Surabaya",
    "Semarang",
    "Yogyakarta",
    "Medan",
    "Makassar",
    "Denpasar",
    "Bali",
    "Belitung",
    "Malang",
    "Bogor",
    "Depok",
    "Tangerang",
    "Bekasi",
    "Palembang",
    "Balikpapan",
    "Manado",
    "Beijing",
    "Tokyo",
    "New York",
    "London",
    "Paris",
    "Indonesia",
    "Brazil",
    "Prancis",
    "Korea",
    "Belanda",
    "Inggris",
    "Kuala Lumpur"
]

organisasi = [
    "PT Astra International",
    "Universitas Indonesia",
    "Bank Mandiri",
    "Kementerian Keuangan",
    "PBB",
    "Tesla",
    "Meta",
    "Google",
    "Kompas",
    "RS Siloam",
    "KPK",
    "DPR RI",
    "Bulog",
    # Organisasi yang NAMANYA mengandung nama orang/tokoh (kata yang juga ada
    # di nama_indonesia/nama_ganda_jalan_orang) - ditambahkan supaya model
    # belajar bahwa nama tokoh yang jadi BAGIAN dari nama lembaga tidak boleh
    # ditandai PERSON secara terpisah (sumber False Positive yang ditemukan
    # saat eval manual: "Yayasan Ahmad Dahlan", "Universitas Diponegoro").
    "Yayasan Ahmad Dahlan",
    "Universitas Diponegoro",
    "Rumah Sakit Bunda",
    "Sekolah Dasar Kartini",
    "Bank Woori Saudara",
]

# Hanya frasa/istilah yang gramatikal apa adanya (tanggal, kontak, rujukan
# hukum, mata uang) - kata jabatan/profesi dipisah ke variabel `jabatan`
# di bawah karena butuh template tersendiri yang menaruhnya persis di
# depan nama (pola appositive: "Gubernur DKI Jakarta, Anies Baswedan, ...").
istilah_lain = [
    "APBN 2025",
    "UU No. 8 Tahun 1999",
    "johndoe@example.com",
    "0812-3456-7890",
    "19.00 WIB",
    "Rp1.000.000",
    "17 Agustus 1945",
    "COVID-19",
    "Yth. Bapak/Ibu",
]

bukan_person = tempat + organisasi + istilah_lain

# Jabatan/profesi: dipakai khusus di template_jabatan supaya model belajar
# pola "Jabatan [, ]Nama[, ]..." tanpa ikut menandai jabatannya sebagai PERSON.
jabatan = [
    "Gubernur DKI Jakarta",
    "Wali Kota Surabaya",
    "Menteri Keuangan RI",
    "Menteri Luar Negeri",
    "Presiden AS ke-46",
    "Perdana Menteri Inggris",
    "CEO Tesla",
    "Direktur Utama",
    "Ketua DPR RI",
    "Kepala Sekolah",
    "Rektor Universitas Indonesia",
    "Panglima TNI",
    "Kapolri",
    "Jaksa Agung",
    "Hakim Ketua",
    "Penyanyi",
    "Pelukis",
    "Pembawa Acara",
]

# 4e. Kata kapital-di-awal-kalimat yang BUKAN nama orang tapi rawan salah
#    tertandai PERSON gara-gara posisinya sebagai subjek kalimat (persis
#    posisi {nama} di banyak template_kalimat). Ditemukan lewat eval manual
#    ("Yang hadir...", "Atlet bulu tangkis...", "Bunga di taman...", "Halo
#    semua...", "Kost berada di...") - model jadi overgeneralisasi "kata
#    kapital tunggal di awal kalimat = PERSON" setelah pool nama tunggal
#    diperbanyak. Dipakai di template_kata_ambigu di bawah sebagai negative
#    murni supaya model belajar membedakan lewat konteks kalimat, bukan
#    sekadar posisi/kapitalisasi token.
kata_ambigu_bukan_nama = [
    "Bunga",
    "Mawar",
    "Fajar",
    "Kost",
    "Halo",
    "Atlet",
    "Yang",
    "Lengkap",
    "Mereka",
    "Kami",
]

template_kata_ambigu = [
    "{kata} mekar dengan indah di taman setiap pagi.",
    "{kata} mulai terlihat sejak dini hari.",
    "{kata} bukan bagian dari agenda rapat kali ini.",
    "{kata} hadir dalam acara itu belum dikonfirmasi panitia.",
    "{kata} berada di lokasi yang sudah ditentukan.",
    "{kata} semua diundang untuk hadir dalam acara tersebut.",
    "{kata}, menurut laporan itu, akan segera diperbarui.",
]

# 4f. Headline ALL-CAPS TANPA nama orang sama sekali - eval manual menemukan
#    model salah menandai SELURUH headline sebagai PERSON ("GEMPA BUMI
#    GUNCANG WILAYAH SELATAN JAWA" tertandai PER penuh). Augmentasi ALL-CAPS
#    acak (PELUANG_ALL_CAPS) sejauh ini kebanyakan mengenai kalimat berisi
#    nama, jadi contoh headline negatif (tanpa PERSON) kurang terwakili.
#    Ditulis literal huruf besar (bukan lewat .upper()) supaya selalu masuk
#    tanpa bergantung peluang acak.
template_headline_negatif = [
    "GEMPA BUMI GUNCANG WILAYAH SELATAN JAWA.",
    "HARGA BBM RESMI NAIK MULAI BESOK.",
    "BANJIR BESAR RENDAM RIBUAN RUMAH DI PESISIR.",
    "EKONOMI NASIONAL TUMBUH 5 PERSEN KUARTAL INI.",
    "KEBAKARAN HUTAN MELUAS DI KALIMANTAN.",
    "INFLASI TAHUNAN TURUN KE LEVEL TERENDAH.",
    "CUACA EKSTREM DIPERKIRAKAN LANDA JAKARTA PEKAN INI.",
    "PEMERINTAH UMUMKAN KEBIJAKAN SUBSIDI BARU.",
]


## Komponen Alamat

In [8]:

# 4b. Komponen penyusun alamat Indonesia (dikomposisi acak jadi 1 string
#    alamat lengkap oleh buat_alamat()). Kelengkapan bervariasi supaya
#    model belajar mengenali alamat pendek maupun sangat lengkap.
jenis_jalan = ["Jl.", "Jalan", "Gang", "Gg."]

# Nama jalan generik (bukan nama orang) - aman jadi ALAMAT tanpa ambigu.
nama_jalan_umum = [
    "Melati",
    "Mawar",
    "Merdeka",
    "Kenanga",
    "Cendrawasih",
    "Anggrek",
    "Kebon Jeruk",
    "Cempaka Putih",
    "Flamboyan",
    "Mangga Dua",
    "Cemara",
    "Rajawali",
]

# Nama jalan yang berasal dari nama tokoh (lihat nama_ganda_jalan_orang) -
# inilah sumber ambiguitas "Jl. X" (ALAMAT) vs "X" saja (PER).
nama_jalan_dari_orang = nama_ganda_jalan_orang

kelurahan_desa = [
    "Kel. Menteng",
    "Kel. Sukajadi",
    "Kel. Cihapit",
    "Kel. Gubeng",
    "Desa Sukamaju",
    "Desa Cibogo",
    "Kel. Antapani",
    "Kel. Tegalsari",
]

kecamatan = [
    "Kec. Menteng",
    "Kec. Cimahi Utara",
    "Kec. Buah Batu",
    "Kec. Sukasari",
    "Kec. Gubeng",
    "Kec. Tegalsari",
    "Kec. Coblong",
]

# Kota/kabupaten khusus konteks alamat (subset Indonesia dari `tempat`,
# tanpa negara/kota luar negeri supaya alamatnya realistis).
kota_alamat = [
    "Jakarta",
    "Bandung",
    "Surabaya",
    "Semarang",
    "Yogyakarta",
    "Medan",
    "Makassar",
    "Denpasar",
    "Malang",
    "Bogor",
    "Depok",
    "Tangerang",
    "Bekasi",
    "Palembang",
    "Balikpapan",
    "Manado",
]

provinsi = [
    "DKI Jakarta",
    "Jawa Barat",
    "Jawa Tengah",
    "Jawa Timur",
    "DI Yogyakarta",
    "Sumatera Utara",
    "Sulawesi Selatan",
    "Bali",
    "Kalimantan Timur",
]


def buat_kode_pos():
    return str(random.randint(10000, 99999))


def buat_alamat():
    """Komposisi alamat dgn kelengkapan acak: pendek/sedang/lengkap."""
    nama_jalan = random.choice(nama_jalan_umum + nama_jalan_dari_orang)
    jalan = f"{random.choice(jenis_jalan)} {nama_jalan}"
    if random.random() < 0.7:
        nomor = random.randint(1, 200)
        sufiks = random.choice(["", "A", "B"])
        jalan += f" No. {nomor}{sufiks}"

    bagian = [jalan]
    kelengkapan = random.choice(["pendek", "sedang", "lengkap"])

    if kelengkapan in ("sedang", "lengkap"):
        rt = random.randint(1, 20)
        rw = random.randint(1, 20)
        bagian.append(f"RT {rt:02d}/RW {rw:02d}")
        bagian.append(random.choice(kelurahan_desa))

    if kelengkapan == "lengkap":
        bagian.append(random.choice(kecamatan))

    bagian.append(random.choice(kota_alamat))

    if kelengkapan == "lengkap":
        bagian.append(random.choice(provinsi))
        bagian.append(buat_kode_pos())

    return ", ".join(bagian)


## Komponen Alamat Internasional

In [9]:

# 4c. Komponen alamat luar negeri: format berbeda dari alamat Indonesia
#    (nomor di depan nama jalan, kode pos per negara, tanpa RT/RW/kelurahan).
HURUF_KAPITAL = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"

jenis_jalan_internasional = ["Street", "Avenue", "Road", "Boulevard", "Lane", "Drive"]

# Nama jalan generik ala Barat (bukan nama orang) - aman jadi ADR tanpa ambigu.
nama_jalan_internasional_umum = [
    "Oxford",
    "Baker",
    "Sunset",
    "Wall",
    "King",
    "Queen",
    "Park",
    "Bridge",
    "Orchard",
    "Victoria",
]

# Nama jalan dari nama tokoh (lihat nama_ganda_jalan_orang_internasional) -
# sumber ambiguitas "123 Washington Street" (ADR) vs "Washington" saja (PER).
nama_jalan_internasional_dari_orang = nama_ganda_jalan_orang_internasional

# (kota, negara) - dipasangkan supaya konsisten & format kode pos sesuai negara.
kota_internasional = [
    ("New York", "Amerika Serikat"),
    ("Los Angeles", "Amerika Serikat"),
    ("London", "Inggris"),
    ("Paris", "Prancis"),
    ("Berlin", "Jerman"),
    ("Tokyo", "Jepang"),
    ("Seoul", "Korea Selatan"),
    ("Sydney", "Australia"),
    ("Toronto", "Kanada"),
    ("Singapura", "Singapura"),
    ("Kuala Lumpur", "Malaysia"),
    ("Amsterdam", "Belanda"),
    ("Dubai", "Uni Emirat Arab"),
]


def buat_kode_pos_internasional(negara):
    """Format kode pos disesuaikan per negara supaya alamatnya realistis."""
    if negara == "Amerika Serikat":
        return str(random.randint(10000, 99999))
    if negara == "Inggris":
        return (
            f"{random.choice(HURUF_KAPITAL)}{random.choice(HURUF_KAPITAL)}"
            f"{random.randint(1, 9)} {random.randint(1, 9)}"
            f"{random.choice(HURUF_KAPITAL)}{random.choice(HURUF_KAPITAL)}"
        )
    if negara == "Kanada":
        return (
            f"{random.choice(HURUF_KAPITAL)}{random.randint(0, 9)}{random.choice(HURUF_KAPITAL)} "
            f"{random.randint(0, 9)}{random.choice(HURUF_KAPITAL)}{random.randint(0, 9)}"
        )
    return str(random.randint(10000, 99999))


def buat_alamat_internasional():
    """Komposisi alamat luar negeri: '<no> <jalan> <jenis>, <kota>[, <kode pos>], <negara>'."""
    kota, negara = random.choice(kota_internasional)
    nama_jalan = random.choice(nama_jalan_internasional_umum + nama_jalan_internasional_dari_orang)
    nomor = random.randint(1, 999)
    jalan = f"{nomor} {nama_jalan} {random.choice(jenis_jalan_internasional)}"

    bagian = [jalan, kota]
    if random.random() < 0.6:
        bagian.append(buat_kode_pos_internasional(negara))
    bagian.append(negara)

    return ", ".join(bagian)


def buat_alamat_acak():
    """Dispatcher: campuran format domestik umum, internasional, dan variasi
    khusus (kost/dusun, perumahan, apartemen, PO Box) - lihat cell 4d untuk
    fungsi buat_alamat_kost_dusun/buat_alamat_perumahan/buat_alamat_apartemen/
    buat_po_box yang menambah format yang sebelumnya belum terwakili."""
    r = random.random()
    if r < 0.55:
        return buat_alamat()
    elif r < 0.75:
        return buat_alamat_internasional()
    elif r < 0.85:
        return buat_alamat_kost_dusun()
    elif r < 0.93:
        return buat_alamat_perumahan()
    elif r < 0.98:
        return buat_alamat_apartemen()
    else:
        return buat_po_box()


In [10]:

# 4d. Variasi format ADR tambahan: kost/dusun (pedesaan, tanpa "Jl."),
#    perumahan/kompleks, apartemen, dan PO Box. Ditambahkan karena eval
#    manual menemukan format-format ini belum dikenali model - sebelumnya
#    hanya format "Jl./Jalan ... No. ..." yang terwakili di training.
dusun_list = [
    "Dusun Krajan",
    "Dusun Sumber",
    "Dusun Karangasem",
    "Dusun Wonosari",
    "Dusun Tegalrejo",
    "Dusun Ngepoh",
]

kabupaten_list = [
    "Kab. Bandung",
    "Kab. Sleman",
    "Kab. Sidoarjo",
    "Kab. Bogor",
    "Kab. Malang",
    "Kab. Cirebon",
]

nama_perumahan = [
    "Griya Asri",
    "Bukit Indah",
    "Taman Sari",
    "Puri Nirwana",
    "Villa Melati",
    "Grand Wisata",
]

nama_apartemen = [
    "Kalibata City",
    "Gading Nias Residences",
    "Green Pramuka City",
    "Taman Anggrek Residence",
    "Kemang Village",
]

tower_apartemen = ["A", "B", "C", "D", "Mawar", "Melati"]


def buat_alamat_kost_dusun():
    """Format pedesaan: 'Dusun X RT../RW.., Desa/Kel Y, Kec. Z, Kab. W' - tanpa 'Jl.'."""
    dusun = random.choice(dusun_list)
    rt = random.randint(1, 20)
    rw = random.randint(1, 20)
    desa = random.choice(kelurahan_desa)
    kec = random.choice(kecamatan)
    kab = random.choice(kabupaten_list)
    return f"{dusun} RT {rt:02d}/RW {rw:02d}, {desa}, {kec}, {kab}"


def buat_alamat_perumahan():
    """Format kompleks perumahan: 'Perumahan X Blok Y No. Z, Kota'."""
    nama = random.choice(nama_perumahan)
    blok = f"{random.choice(HURUF_KAPITAL[:8])}{random.randint(1, 20)}"
    nomor = random.randint(1, 50)
    kota = random.choice(kota_alamat)
    return f"Perumahan {nama} Blok {blok} No. {nomor}, {kota}"


def buat_alamat_apartemen():
    """Format apartemen: 'Apartemen X Tower Y Lantai Z, Kota'."""
    nama = random.choice(nama_apartemen)
    tower = random.choice(tower_apartemen)
    lantai = random.randint(1, 30)
    kota = random.choice(kota_alamat)
    return f"Apartemen {nama} Tower {tower} Lantai {lantai}, {kota}"


def buat_po_box():
    """Format PO Box: 'PO Box <nomor>, Kota <kode_pos>'."""
    nomor = random.randint(100, 9999)
    kota = random.choice(kota_alamat)
    return f"PO Box {nomor}, {kota} {buat_kode_pos()}"


## Template Kalimat

In [11]:
# 3. Template Kalimat (Bervariasi: berita, kasual, formal, interogatif)
template_kalimat = [
    # Kasual / Sehari-hari
    "Kemarin saya bertemu dengan {nama} di stasiun.",
    "Tolong sampaikan pesan ini kepada {nama} secepatnya.",
    "Saya melihat {nama} sedang makan siang di kantin.",
    "Apakah kamu sudah menghubungi {nama} hari ini?",
    "Kemarin mobilnya {nama} mogok di jalan tol.",
    "Jangan lupa undang {nama} ke acara ulang tahun minggu depan.",
    "Saya punya saudara di jakarta namanya {nama}.",
    "{nama} adalah gubernur jawa barat.",
    "{nama} berangkat ke kantor pagi ini.",
    # Sapaan informal / media sosial - "Halo semua, ..." sempat membuat
    # kata "Halo" ikut tertandai PERSON; template ini mengajarkan bahwa
    # hanya nama setelah "nama gue" yang berlabel PERSON.
    "Halo semua, nama gue {nama}, salam kenal ya!",
    # Formal / Bisnis
    "Proyek ini dipimpin langsung oleh {nama}.",
    "Menurut {nama}, kebijakan ini harus segera dievaluasi.",
    "Rapat dewan direksi besok akan dipimpin oleh {nama}.",
    "Kami sedang menunggu tanda tangan dari {nama} untuk mencairkan dana.",
    "Laporan tersebut telah dikirimkan langsung ke meja {nama}.",
    "Keputusan final berada di tangan {nama} selaku manajer proyek.",
    # Berita / Jurnalistik
    "{nama} adalah salah satu tokoh penting dalam sejarah.",
    "Apakah benar {nama} akan datang ke konferensi besok?",
    "Buku terbaru itu ditulis oleh {nama} dan diterbitkan bulan lalu.",
    "Dalam konferensi persnya, {nama} menolak memberikan komentar.",
    "Pihak kepolisian masih memeriksa {nama} sebagai saksi utama.",
    "Penghargaan Nobel tahun ini jatuh kepada {nama} atas dedikasinya.",
    "Karya seni milik {nama} berhasil terjual dengan harga fantastis.",
    # Akademis / Opini
    "Gagasan yang dikemukakan oleh {nama} memicu perdebatan panjang.",
    "Banyak pihak yang setuju dengan analisis {nama} mengenai krisis ekonomi.",
    "Teori yang diusulkan oleh {nama} telah dipatahkan oleh penemuan baru.",
    # Atribusi dialog / kutipan - {nama} sengaja diletakkan di berbagai posisi
    # (awal, tengah, akhir) supaya model tidak hanya mengenali nama yang
    # muncul di akhir kalimat setelah tanda kutip.
    '"Saya akan segera berangkat," ujar {nama}.',
    'Menurut {nama}, "keputusan ini sudah final."',
    '"Ini adalah langkah yang tepat," tutur {nama} kepada wartawan.',
    '{nama} menegaskan, "kami akan terus berusaha."',
    '"Terima kasih atas dukungannya," ucap {nama} sambil tersenyum.',
]

# 3b. Template Media Sosial: nama dipakai sebagai handle huruf kecil tanpa
#    spasi ("@budisantoso") - format ini belum terwakili di training
#    sebelumnya (semua contoh selalu huruf kapital di awal kata), sehingga
#    model gagal mengenali handle media sosial sebagai PERSON.
template_medsos = [
    "{handle} baru saja mengunggah foto liburannya.",
    "{handle} membagikan update terbaru di akunnya.",
    "Jangan lupa follow {handle} untuk info terkini.",
]

In [ ]:
# 5b. Template Hard-Negative: nama + distraktor dalam satu kalimat
#     (hanya {nama} yang ditandai PERSON, {lain} sengaja TIDAK ditandai)
template_gabungan = [
    "{nama} bertemu dengan rekannya di {lain}.",
    "{nama} bekerja di {lain}.",
    "Menurut {nama}, {lain} perlu dievaluasi ulang.",
    "{nama} tinggal di {lain} sejak tahun lalu.",
    "Kemarin {nama} membahas {lain} bersama timnya.",
    "{nama} baru saja pindah dari {lain}.",
    "Nomor kontak {nama} bisa dihubungi melalui {lain}.",
    "\"{lain} adalah prioritas kita,\" kata {nama}.",
    "{nama} menghadiri rapat mengenai {lain} pagi ini.",
    "Surat dari {nama} membahas rencana terkait {lain}.",
    # Pola "bertemu dengan {nama} di {lain}" - ditambahkan karena eval manual
    # menemukan model menggabungkan nama dengan lokasi setelahnya jadi satu
    # span PERSON ("Budi Santoso di Jakarta" tertandai sebagai satu entity).
    # Sebelumnya template serupa (template_kalimat) selalu memakai kata
    # generik tetap ("stasiun"), tidak pernah nama tempat asli di posisi ini.
    "Kemarin saya bertemu dengan {nama} di {lain}.",
    "Surat itu dikirim oleh {nama} di {lain} minggu lalu.",
]

# 5c. Template Negative murni: TIDAK ada PERSON sama sekali (entitas kosong)
template_negatif = [
    "Kemarin saya pergi ke {lain} untuk urusan pekerjaan.",
    "{lain} baru saja merilis laporan tahunan.",
    "Rapat membahas {lain} akan dilaksanakan besok.",
    "Nomor kontak yang bisa dihubungi adalah {lain}.",
    "Acara tersebut diselenggarakan di {lain}.",
    "Menurut laporan, {lain} mengalami peningkatan signifikan.",
    "Kantor pusatnya berlokasi di {lain}.",
    "Dokumen itu merujuk pada {lain}.",
]

# 5d. Template Jabatan+Nama (appositive): jabatan/profesi ditempatkan PERSIS
#     di depan/belakang nama, meniru pola berita yang paling sering salah
#     ("Gubernur DKI Jakarta Anies Baswedan...", "Wali Kota Surabaya, Eri
#     Cahyadi, ..."). Hanya {nama} yang berlabel PERSON.
template_jabatan = [
    "{jabatan} {nama} meresmikan acara tersebut.",
    "{jabatan}, {nama}, meresmikan acara tersebut.",
    "{jabatan} {nama} menyampaikan pernyataan resmi kemarin.",
    "{jabatan}, {nama}, menyampaikan pernyataan resmi kemarin.",
    "Menurut {jabatan} {nama}, kebijakan ini akan segera direvisi.",
    "{nama}, selaku {jabatan}, hadir dalam konferensi pers.",
    "{jabatan} sebelumnya adalah {nama}.",
    "{nama} resmi menjabat sebagai {jabatan} mulai bulan depan.",
]

# 5e. Template Multi-Nama: enumerasi 2-4 nama dalam satu kalimat, meniru
#     pola daftar peserta/tokoh yang sering muncul di teks berita.
template_multi_2 = [
    "Rapat tersebut dihadiri oleh {nama1} dan {nama2}.",
    "{nama1} dan {nama2} kompak hadir dalam acara itu.",
    "Menurut {nama1} dan {nama2}, kebijakan ini perlu direvisi.",
    "Baik {nama1} maupun {nama2} belum memberikan komentar.",
]
template_multi_3 = [
    "Di antara peserta terdapat {nama1}, {nama2}, dan {nama3}.",
    "{nama1}, {nama2}, serta {nama3} kompak hadir dalam acara itu.",
    "Panitia mengundang {nama1}, {nama2}, dan {nama3} sebagai narasumber.",
    "Tim tersebut beranggotakan {nama1}, {nama2}, dan {nama3}.",
    # "Yang hadir..." - kata relatif "Yang" di awal kalimat sempat ikut
    # tertandai PERSON oleh model; template ini mengajarkan bahwa hanya nama
    # setelah "adalah" yang berlabel PERSON, "Yang hadir..." di depan tidak.
    "Yang hadir dalam pertemuan itu adalah {nama1}, {nama2}, dan {nama3}.",
    # "meliputi X, Y, dan Z dari ..." - ditemukan lewat eval manual bikin
    # 2 nama pertama tergabung jadi satu span ("Dedi, Fitri" jadi 1 entity).
    "Peserta pelatihan meliputi {nama1}, {nama2}, dan {nama3} dari kantor cabang.",
    "Daftar penerima meliputi {nama1}, {nama2}, dan {nama3} dari divisi terkait.",
]

# Enumerasi 4 nama - eval manual menemukan model gagal memisahkan daftar
# sepanjang ini ("Bambang, Wati, Joko, dan Lestari" tergabung sebagian jadi
# satu span) karena generator sebelumnya cuma pernah melatih 2-3 nama.
template_multi_4 = [
    "Rapat dihadiri oleh {nama1}, {nama2}, {nama3}, dan {nama4} sore ini.",
    "Panitia mengundang {nama1}, {nama2}, {nama3}, dan {nama4} sebagai narasumber.",
    "Tim tersebut beranggotakan {nama1}, {nama2}, {nama3}, dan {nama4}.",
    "Di antara peserta terdapat {nama1}, {nama2}, {nama3}, dan {nama4}.",
]

In [13]:
# 5f. Template Alamat: kalimat dengan tepat satu ALAMAT (hasil buat_alamat()).
template_alamat = [
    "Alamat saya di {alamat}.",
    "Silakan kirim paket ke {alamat}.",
    "Rumah beliau berada di {alamat}.",
    "KTP menunjukkan alamat di {alamat}.",
    "Kantor kami beralamat di {alamat}.",
    "Rapat RT akan diadakan di {alamat}.",
    "Ia baru saja pindah ke {alamat}.",
    "Sesuai catatan sipil, domisili tercatat di {alamat}.",
    # "Kost berada di ..." - kata "Kost" di awal kalimat sempat ikut
    # tertandai PERSON; template ini mengajarkan konteksnya (diikuti ADR).
    "Kost berada di {alamat}.",
    "Rumah kost berada di {alamat}.",
]

# 5g. Template Negative khusus Alamat: cuma sebut kota/kecamatan SENDIRIAN,
#     BUKAN alamat terstruktur - harus TETAP kosong (bukan ALAMAT).
template_alamat_negatif = [
    "Saya tinggal di {kota} sejak kecil.",
    "Ia baru saja pindah ke {kota}.",
    "Kantor cabang baru akan dibuka di {kota}.",
    "Kami sudah lama menetap di {kota}.",
    "Acara tahunan itu selalu digelar di {kota}.",
]

# 5h. Template Kombinasi: PER dan ALAMAT sekaligus dalam satu kalimat,
#     meniru pola form/catatan ("Nama: ..., Alamat: ...").
template_kombinasi = [
    "{nama} tinggal di {alamat}.",
    "Paket untuk {nama} dikirim ke {alamat}.",
    "Nama: {nama}, Alamat: {alamat}.",
    "{nama} baru saja pindah ke {alamat}.",
    "KTP atas nama {nama} beralamat di {alamat}.",
    "Surat untuk {nama} dialamatkan ke {alamat}.",
    # Variasi label form lain ("Nama Lengkap:", "Penerima:"/"Tujuan:") -
    # ditemukan lewat eval manual bikin model gagal memisahkan PER dari ADR
    # (keduanya malah tergabung jadi satu span) karena wording persis ini
    # belum pernah terwakili di training.
    "Nama Lengkap: {nama}\nAlamat: {alamat}.",
    "Penerima: {nama}, Tujuan: {alamat}",
]

## Generate Data Train CSV

In [ ]:
# 6. Proses Generate Data (campuran PER dan ADR: positif, hard-negatif,
#    jabatan+nama, multi-nama, alamat (domestik+internasional),
#    alamat-negatif, kombinasi, negatif murni, kata-ambigu-negatif, medsos,
#    headline-negatif)
semua_nama = (
    nama_indonesia
    + nama_internasional
    + nama_dengan_gelar
    # nama_ganda_jalan_orang dimasukkan 2x supaya proporsinya lebih besar -
    # eval manual menemukan recall utk nama dual-use jalan/tokoh (Diponegoro,
    # Sukarno) sedikit turun setelah kata_ambigu_negatif ditambahkan (model
    # jadi lebih ragu2 pada nama tunggal di posisi subjek kalimat). AMAN
    # dipakai di sini karena kategori yang memakai `semua_nama` langsung
    # (positif/hard_negatif/jabatan_nama/kombinasi/medsos) hanya butuh 1
    # nama per kalimat via random.choice - duplikat nilai tidak masalah.
    + nama_ganda_jalan_orang
    + nama_ganda_jalan_orang
    + nama_ganda_jalan_orang_internasional
)
# Khusus utk multi_nama (random.sample, butuh 2-4 nama BERBEDA dalam satu
# kalimat): TIDAK boleh pakai `semua_nama` yang sengaja diduplikasi di atas -
# random.sample menjamin index unik, tapi tidak menjamin STRING-nya unik
# kalau ada 2 index berbeda dengan nilai sama persis. Kalau nama1==nama2,
# cari_semua_span mencari string yang sama dua kali dan menghasilkan span
# yang identik/tumpang tindih -> doc.ents menolak saat convert ke .spacy
# (ValueError E1010: token termasuk lebih dari satu span).
semua_nama_unik = list(dict.fromkeys(semua_nama))
# Provinsi ditambahkan supaya "provinsi sendirian" (mis. "Jawa Barat") juga
# diajarkan sebagai negative ADR - eval manual menemukan model salah
# menandai provinsi tunggal sebagai ADR.
kota_negatif_alamat = (
    kota_alamat + provinsi + [kota for kota, _negara in kota_internasional]
)
total_data_yang_dibuat = 3000
proporsi = {
    "positif": 0.20,
    "hard_negatif": 0.10,
    "jabatan_nama": 0.10,
    "multi_nama": 0.10,
    "alamat": 0.14,
    "alamat_negatif": 0.05,
    "kombinasi": 0.13,
    "negatif": 0.05,
    "kata_ambigu_negatif": 0.05,
    "medsos": 0.05,
    "headline_negatif": 0.03,
}
# Peluang sebuah kalimat diubah jadi HURUF KAPITAL SEMUA (gaya headline
# berita). str.upper() tidak mengubah panjang karakter untuk teks kita
# (Latin + diakritik), jadi posisi start/end entity yang sudah dihitung
# dari teks asli tetap valid dipakai pada versi kapitalnya.
PELUANG_ALL_CAPS = 0.08
dataset = []


def cari_semua_span(teks, daftar_entitas):
    """daftar_entitas: list (nilai, label). Dicari independen lalu diurutkan
    berdasarkan posisi kemunculan - tidak bergantung urutan placeholder di
    template, sehingga aman untuk kombinasi PER+ALAMAT dalam urutan apa pun.
    Span yang tumpang tindih dengan span sebelumnya (sudah diterima) DIBUANG
    - jaga-jaga andai ada nilai entitas yang saling beririsan (mis. dua nama
    yang kebetulan sama, atau satu nilai adalah substring nilai lain) supaya
    tidak menghasilkan doc.ents yang ditolak spaCy (ValueError E1010)."""
    entitas = []
    for nilai, label in daftar_entitas:
        match = re.search(re.escape(nilai), teks)
        if match:
            entitas.append({"start": match.start(), "end": match.end(), "label": label})
    entitas.sort(key=lambda e: e["start"])
    hasil = []
    akhir_terakhir = -1
    for e in entitas:
        if e["start"] >= akhir_terakhir:
            hasil.append(e)
            akhir_terakhir = e["end"]
    return hasil


for _ in range(total_data_yang_dibuat):
    jenis = random.choices(
        population=list(proporsi.keys()),
        weights=list(proporsi.values()),
        k=1,
    )[0]

    entitas = []

    if jenis == "positif":
        # Kalimat dengan tepat satu PERSON, tanpa distraktor
        nama_terpilih = random.choice(semua_nama)
        template_terpilih = random.choice(template_kalimat)
        teks_lengkap = template_terpilih.format(nama=nama_terpilih)
        entitas = cari_semua_span(teks_lengkap, [(nama_terpilih, LABEL_PERSON)])

    elif jenis == "hard_negatif":
        # Kalimat dengan PERSON + distraktor (tempat/organisasi/istilah).
        # Hanya nama yang ditandai, distraktor sengaja dibiarkan tidak berlabel
        # supaya model belajar tidak ikut menandainya sebagai PERSON.
        nama_terpilih = random.choice(semua_nama)
        lain_terpilih = random.choice(bukan_person)
        template_terpilih = random.choice(template_gabungan)
        teks_lengkap = template_terpilih.format(nama=nama_terpilih, lain=lain_terpilih)
        entitas = cari_semua_span(teks_lengkap, [(nama_terpilih, LABEL_PERSON)])

    elif jenis == "jabatan_nama":
        # Kalimat appositive: jabatan/profesi persis di depan/belakang nama.
        # Hanya nama yang berlabel PERSON, jabatannya tidak.
        nama_terpilih = random.choice(semua_nama)
        jabatan_terpilih = random.choice(jabatan)
        template_terpilih = random.choice(template_jabatan)
        teks_lengkap = template_terpilih.format(nama=nama_terpilih, jabatan=jabatan_terpilih)
        entitas = cari_semua_span(teks_lengkap, [(nama_terpilih, LABEL_PERSON)])

    elif jenis == "multi_nama":
        # Kalimat dengan 2-4 PERSON sekaligus (enumerasi/daftar nama).
        # Pakai semua_nama_unik (bukan semua_nama) supaya nama1..nama4
        # dijamin BERBEDA satu sama lain - lihat penjelasan di atas.
        jumlah_nama = random.choice([2, 3, 4])
        nama_list = random.sample(semua_nama_unik, jumlah_nama)
        if jumlah_nama == 2:
            template_terpilih = random.choice(template_multi_2)
            teks_lengkap = template_terpilih.format(nama1=nama_list[0], nama2=nama_list[1])
        elif jumlah_nama == 3:
            template_terpilih = random.choice(template_multi_3)
            teks_lengkap = template_terpilih.format(
                nama1=nama_list[0], nama2=nama_list[1], nama3=nama_list[2]
            )
        else:
            template_terpilih = random.choice(template_multi_4)
            teks_lengkap = template_terpilih.format(
                nama1=nama_list[0], nama2=nama_list[1], nama3=nama_list[2], nama4=nama_list[3]
            )
        entitas = cari_semua_span(teks_lengkap, [(n, LABEL_PERSON) for n in nama_list])

    elif jenis == "alamat":
        # Kalimat dengan tepat satu ADR (domestik atau internasional), tanpa nama orang.
        alamat_terpilih = buat_alamat_acak()
        template_terpilih = random.choice(template_alamat)
        teks_lengkap = template_terpilih.format(alamat=alamat_terpilih)
        entitas = cari_semua_span(teks_lengkap, [(alamat_terpilih, LABEL_ADDRESS)])

    elif jenis == "alamat_negatif":
        # Cuma sebut nama kota sendirian (domestik ATAU internasional),
        # bukan alamat terstruktur - harus TETAP tidak berlabel ADR.
        kota_terpilih = random.choice(kota_negatif_alamat)
        template_terpilih = random.choice(template_alamat_negatif)
        teks_lengkap = template_terpilih.format(kota=kota_terpilih)
        # entitas tetap kosong []

    elif jenis == "kombinasi":
        # PER dan ADR sekaligus dalam satu kalimat (pola form/catatan).
        nama_terpilih = random.choice(semua_nama)
        alamat_terpilih = buat_alamat_acak()
        template_terpilih = random.choice(template_kombinasi)
        teks_lengkap = template_terpilih.format(nama=nama_terpilih, alamat=alamat_terpilih)
        entitas = cari_semua_span(
            teks_lengkap,
            [(nama_terpilih, LABEL_PERSON), (alamat_terpilih, LABEL_ADDRESS)],
        )

    elif jenis == "kata_ambigu_negatif":
        # Kata kapital-di-awal-kalimat yang MIRIP posisi nama tapi BUKAN
        # PERSON (lihat cell 4e) - selalu entitas kosong.
        kata_terpilih = random.choice(kata_ambigu_bukan_nama)
        template_terpilih = random.choice(template_kata_ambigu)
        teks_lengkap = template_terpilih.format(kata=kata_terpilih)
        # entitas tetap kosong []

    elif jenis == "medsos":
        # Nama sebagai handle media sosial huruf kecil tanpa spasi ("@budisantoso").
        # "@" sengaja TIDAK ikut jadi bagian span PERSON - konsisten dengan
        # konvensi bahwa simbol handle bukan bagian dari nama itu sendiri.
        nama_terpilih = random.choice(semua_nama)
        username = re.sub(r"[^a-z0-9]", "", nama_terpilih.lower())
        handle = "@" + username
        template_terpilih = random.choice(template_medsos)
        teks_lengkap = template_terpilih.format(handle=handle)
        entitas = cari_semua_span(teks_lengkap, [(username, LABEL_PERSON)])

    elif jenis == "headline_negatif":
        # Headline ALL-CAPS tanpa nama orang sama sekali (lihat cell 4f).
        teks_lengkap = random.choice(template_headline_negatif)
        # entitas tetap kosong []

    else:  # negatif murni: sama sekali tidak ada entity apa pun di kalimat
        lain_terpilih = random.choice(bukan_person)
        template_terpilih = random.choice(template_negatif)
        teks_lengkap = template_terpilih.format(lain=lain_terpilih)
        # entitas tetap kosong []

    # Augmentasi HURUF KAPITAL SEMUA (gaya headline) - dilakukan setelah span
    # entity dihitung dari teks asli, karena upper() tidak mengubah panjang
    # karakter sehingga start/end tetap valid.
    if random.random() < PELUANG_ALL_CAPS:
        teks_lengkap = teks_lengkap.upper()

    dataset.append(
        {
            "teks": teks_lengkap,
            "entitas": json.dumps(
                entitas
            ),  # Ubah list/dict ke bentuk string (JSON) untuk CSV
        }
    )

# 7. Simpan ke CSV dengan versioning otomatis (agar file lama tidak tertimpa)
base_nama_file = "dataset_ner_3000"
versi = 1
nama_file = f"{base_nama_file}_v{versi}.csv"
while os.path.exists(nama_file):
    versi += 1
    nama_file = f"{base_nama_file}_v{versi}.csv"

with open(nama_file, mode="w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=["teks", "entitas"])
    writer.writeheader()
    writer.writerows(dataset)

jumlah_per = sum(
    1 for d in dataset if any(e["label"] == LABEL_PERSON for e in json.loads(d["entitas"]))
)
jumlah_alamat = sum(
    1 for d in dataset if any(e["label"] == LABEL_ADDRESS for e in json.loads(d["entitas"]))
)
jumlah_kosong = sum(1 for d in dataset if not json.loads(d["entitas"]))
print(f"Berhasil membuat {len(dataset)} baris data latih di file {nama_file}!")
print(f"  - Baris mengandung {LABEL_PERSON}     : {jumlah_per}")
print(f"  - Baris mengandung {LABEL_ADDRESS} : {jumlah_alamat}")
print(f"  - Baris tanpa entity apa pun : {jumlah_kosong}")